# Félix Insólito — prueba de Wan 2.2 (imagen → vídeo)

Anima una imagen (por ejemplo `ep01-p01-inicio.png`) con **Wan 2.2 TI2V-5B**, el modelo pequeño oficial
(https://github.com/Wan-Video/Wan2.2), en vertical 704×1280 a 24 fps (unos 5 s).

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* **L4** o **A100**.
El modelo necesita al menos **24 GB de memoria de vídeo**; la GPU gratuita T4 (16 GB) no alcanza.

Pasos: ejecuta las celdas en orden (▶). La celda 4 te pedirá subir la imagen.

## 1. Comprobar la GPU

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno a GPU (L4 o A100)."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} — {vram:.1f} GB")
if vram < 22:
    raise SystemExit("Esta GPU tiene menos de 24 GB. Wan 2.2 TI2V-5B no cabe: elige L4 o A100 y vuelve a empezar.")
print("OK: la GPU sirve.")

## 2. Descargar el código oficial e instalar dependencias (5–15 min)

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
# flash_attn se instala al final (así lo recomienda el README oficial)
!grep -v flash_attn requirements.txt > req_sin_flash.txt
!pip install -q -r req_sin_flash.txt
!pip install -q flash_attn --no-build-isolation || echo "AVISO: flash_attn no se instaló; la generación puede fallar."

## 3. Descargar el modelo TI2V-5B (unos minutos)

In [ ]:
!pip install -q -U "huggingface_hub[cli]"
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /content/Wan2.2/Wan2.2-TI2V-5B

## 4. Subir la imagen del plano

In [ ]:
%cd /content
from google.colab import files
subida = files.upload()          # elige p. ej. ep01-p01-inicio.png
IMAGEN = "/content/" + list(subida)[0]
print("Imagen:", IMAGEN)

## 5. Generar el vídeo

Cambia `PROMPT` por el texto de **Kling** del plano en `guiones/ep01-jensen-huang/planos.md`.
`--size 704*1280` = vertical; la proporción sigue a la de la imagen. Tarda varios minutos.

In [ ]:
PROMPT = ("The cartoon news anchor delivers breaking news straight to camera: his mouth moves as he talks, "
          "his handlebar mustache bounces with the words, he raises one eyebrow sarcastically, gives a small "
          "confident nod and taps the desk once with his fingers. Keep the character, outfit and studio exactly "
          "as in the image; smooth premium 3D cartoon animation, no new text, no logos.")
%cd /content/Wan2.2
!python generate.py --task ti2v-5B --size 704*1280 --ckpt_dir ./Wan2.2-TI2V-5B \
    --offload_model True --convert_model_dtype --t5_cpu \
    --image "{IMAGEN}" --prompt "{PROMPT}" --save_file /content/wan22_resultado.mp4

## 6. Descargar el resultado

In [ ]:
import glob, os
from google.colab import files
videos = sorted(glob.glob("/content/*.mp4") + glob.glob("/content/Wan2.2/*.mp4"), key=os.path.getmtime)
assert videos, "No se encontró ningún vídeo: revisa el mensaje de error de la celda 5."
print("Descargando:", videos[-1])
files.download(videos[-1])

Guarda el clip como `clips/ep01/ep01-pNN-wan22.mp4` en el repositorio para compararlo con Kling v3.